# 📖 Notebook 2: Heap-Based Top-K

In Notebook 1, we learned how Count-Min Sketch can **count** items in fixed memory. But counting alone isn't enough — we need to find the **K most popular** items from a stream of millions of events.

The naive approach? Sort all items by count and take the first K. But sorting billions of items is expensive. A **min-heap** lets us maintain the top K items in real time, processing each event in O(log K) time.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why sorting doesn't work for streaming top-K
- How a min-heap efficiently tracks the K largest items
- How to combine Count-Min Sketch + Heap for approximate streaming top-K
- How the SQL-based approach works with PostgreSQL (using indexed ORDER BY + LIMIT)
- The trade-offs between exact (SQL) and approximate (CMS + Heap) approaches

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/top-k
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import heapq
import random
import mmh3

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "topk_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

## 🤔 The Top-K Problem

Given a stream of events, find the K items with the highest count.

### Approach 1: Sort Everything (Naive)

```python
counts = {}  # count every item
for event in stream:
    counts[event] += 1
top_k = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:K]
```

**Problem:** You need to store ALL items in memory AND sort them. With billions of items:
- Storage: 64+ GB for the dictionary
- Sort: O(n log n) where n = billions
- You can't sort until the stream ends

### Approach 2: Min-Heap (Smart)

Keep a heap of size K. For each new item:
1. If the heap has fewer than K items → add it
2. If the item's count > the smallest item in the heap → replace
3. Otherwise → skip it

The heap only holds K items (e.g., 1000), not billions!

```
  Min-Heap (size K=3)        Stream of counts
  ┌───────────┐              
  │     5     │  ← min       vid_042: 5 views   → add to heap
  │   7   9   │              vid_010: 15 views  → add to heap
  └───────────┘              vid_003: 3 views   → skip (3 < min)
                             vid_001: 20 views  → replace min (5)
```

In [ ]:
class TopKHeap:
    """
    Maintains the top K items by count using a min-heap.
    
    The min-heap always keeps the SMALLEST item at the top.
    When a new item has a higher count than the smallest item
    in the heap, we pop the smallest and push the new one.
    
    This means the heap always contains the K LARGEST items seen so far.
    """
    
    def __init__(self, k: int):
        self.k = k
        self.heap = []       # min-heap of (count, item)
        self.item_set = {}   # item -> current count in heap
    
    def push(self, item: str, count: int):
        """
        Consider an item with a given count for the top K.
        """
        if item in self.item_set:
            # Item is already in heap — update it
            # We use lazy deletion: just add the new entry
            # and mark the old one as stale
            self.item_set[item] = count
            heapq.heappush(self.heap, (count, item))
            return
        
        if len(self.item_set) < self.k:
            # Heap not full yet — just add it
            heapq.heappush(self.heap, (count, item))
            self.item_set[item] = count
        elif count > self.heap[0][0]:
            # New item is bigger than the smallest in heap — replace
            # Clean up stale entries first
            self._clean()
            if len(self.item_set) >= self.k:
                old_count, old_item = heapq.heappop(self.heap)
                while old_item in self.item_set and self.item_set[old_item] != old_count:
                    old_count, old_item = heapq.heappop(self.heap)
                if old_item in self.item_set:
                    del self.item_set[old_item]
            heapq.heappush(self.heap, (count, item))
            self.item_set[item] = count
    
    def _clean(self):
        """Remove stale entries from the heap."""
        while self.heap and (
            self.heap[0][1] not in self.item_set or
            self.item_set[self.heap[0][1]] != self.heap[0][0]
        ):
            heapq.heappop(self.heap)
    
    def get_top_k(self) -> list:
        """
        Return the top K items sorted by count (descending).
        """
        result = [(count, item) for item, count in self.item_set.items()]
        result.sort(reverse=True)
        return result[:self.k]

print("✅ TopKHeap class defined!")

In [ ]:
# Simple demo with fruits
topk = TopKHeap(k=3)

fruits = [
    ("apple", 10),
    ("banana", 3),
    ("cherry", 7),
    ("date", 15),
    ("elderberry", 1),
    ("fig", 20),
    ("grape", 12),
]

print("🍎 TopKHeap Demo (K=3)")
print("=" * 50)

for fruit, count in fruits:
    topk.push(fruit, count)
    current = topk.get_top_k()
    display = ", ".join(f"{item}({c})" for c, item in current)
    print(f"  + {fruit}({count}) → Top 3: [{display}]")

print()
print("💡 The heap always contains the 3 largest items seen so far.")
print("   Items with small counts are automatically evicted.")

## 📊 SQL-Based Top-K (Exact, from PostgreSQL)

Before we try the heap approach on real data, let's establish the **ground truth** using SQL.  
Our database already has pre-computed totals in the `video_view_totals` table with an index on `total_views DESC`.

In [ ]:
K = 10

conn = get_db_connection()
cursor = conn.cursor()

# This is the exact query you'd use in production:
# The index on total_views DESC makes this an O(k) operation!
start = time.time()
cursor.execute("""
    SELECT vt.video_id, v.title, vt.total_views
    FROM video_view_totals vt
    JOIN videos v ON v.video_id = vt.video_id
    ORDER BY vt.total_views DESC
    LIMIT %s
""", (K,))
sql_time = (time.time() - start) * 1000

sql_results = cursor.fetchall()
conn.close()

print(f"🏆 Top {K} Videos — SQL Query ({sql_time:.2f} ms)")
print("=" * 65)
print(f"  {'Rank':<6} {'Video ID':<12} {'Title':<35} {'Views':>7}")
print("  " + "-" * 62)
for rank, (vid, title, views) in enumerate(sql_results, 1):
    print(f"  {rank:<6} {vid:<12} {title:<35} {views:>7,}")

print()
print("💡 SQL + indexed column = very fast for pre-computed totals.")
print("   But computing those totals from raw events is the hard part!")

In [ ]:
# Top-K with time window — this is the expensive query
# We need to SUM across hourly buckets and then sort

conn = get_db_connection()
cursor = conn.cursor()

# Top 10 for the last 24 hours
start = time.time()
cursor.execute("""
    SELECT h.video_id, v.title, SUM(h.view_count) as total
    FROM hourly_views h
    JOIN videos v ON v.video_id = h.video_id
    WHERE h.hour_bucket >= NOW() - INTERVAL '24 hours'
    GROUP BY h.video_id, v.title
    ORDER BY total DESC
    LIMIT %s
""", (K,))
window_time = (time.time() - start) * 1000

window_results = cursor.fetchall()
conn.close()

print(f"🏆 Top {K} Videos (Last 24h) — SQL Window Query ({window_time:.2f} ms)")
print("=" * 65)
print(f"  {'Rank':<6} {'Video ID':<12} {'Title':<35} {'Views':>7}")
print("  " + "-" * 62)
for rank, (vid, title, views) in enumerate(window_results, 1):
    print(f"  {rank:<6} {vid:<12} {title:<35} {views:>7,}")

print()
print(f"⏱️  Window query took {window_time:.2f} ms (vs {sql_time:.2f} ms for all-time)")
print("   Window queries need GROUP BY + SUM = much more work for the database.")
print("   At YouTube scale, this would scan hundreds of GB — minutes, not milliseconds!")

## 🔄 Streaming Top-K: CMS + Heap

Now let's combine Count-Min Sketch with our TopKHeap to process view events **as a stream**,  
without needing to store all events or query a database.

The algorithm:
1. A view event arrives: `video_id = "vid_042"`
2. Increment the CMS: `cms.add("vid_042")`
3. Get the estimated count: `est = cms.estimate("vid_042")`
4. Push to the heap: `heap.push("vid_042", est)`
5. The heap automatically keeps only the top K

```
  Event Stream          Count-Min Sketch         Min-Heap (K=5)
  ┌──────────┐         ┌───────────────┐        ┌──────────────┐
  │ vid_042  │──add───▶│ ░░░░█░░░░░░░  │──est──▶│ vid_010: 800 │
  │ vid_010  │         │ ░░░░░░░█░░░░  │        │ vid_001: 500 │
  │ vid_003  │         │ ░░█░░░░░░░░░  │        │ vid_042: 350 │
  │ vid_042  │         └───────────────┘        │ vid_003: 200 │
  │ ...      │                                  │ vid_007: 125 │
  └──────────┘                                  └──────────────┘
```

In [ ]:
# Reuse our CountMinSketch from Notebook 1
class CountMinSketch:
    def __init__(self, width: int, depth: int):
        self.width = width
        self.depth = depth
        self.table = [[0] * width for _ in range(depth)]
    
    def _hash(self, item: str, row: int) -> int:
        return mmh3.hash(item, seed=row) % self.width
    
    def add(self, item: str, count: int = 1):
        for row in range(self.depth):
            col = self._hash(item, row)
            self.table[row][col] += count
    
    def estimate(self, item: str) -> int:
        return min(
            self.table[row][self._hash(item, row)]
            for row in range(self.depth)
        )

print("✅ CountMinSketch class ready")

In [ ]:
# Load view events from the database (simulating a Kafka stream)
conn = get_db_connection()
cursor = conn.cursor()
cursor.execute("SELECT video_id FROM view_events ORDER BY viewed_at")
view_events = [row[0] for row in cursor.fetchall()]
conn.close()

print(f"📦 Loaded {len(view_events):,} view events")
print()

# Process the stream with CMS + Heap
K = 10
cms = CountMinSketch(width=500, depth=5)
topk = TopKHeap(k=K)

start = time.time()
for vid in view_events:
    # Step 1: Count in the sketch
    cms.add(vid)
    # Step 2: Get estimated count
    est = cms.estimate(vid)
    # Step 3: Update the heap
    topk.push(vid, est)
stream_time = (time.time() - start) * 1000

# Get results
heap_results = topk.get_top_k()

# Get exact counts for comparison
exact = {}
for vid in view_events:
    exact[vid] = exact.get(vid, 0) + 1

print(f"🏆 Top {K} Videos — CMS + Heap ({stream_time:.2f} ms)")
print("=" * 70)
print(f"  {'Rank':<6} {'Video ID':<12} {'CMS Est':>9} {'Exact':>7} {'Error':>7}")
print("  " + "-" * 45)

for rank, (est_count, vid) in enumerate(heap_results, 1):
    true_count = exact.get(vid, 0)
    diff = est_count - true_count
    marker = "✅" if diff == 0 else f"+{diff}"
    print(f"  {rank:<6} {vid:<12} {est_count:>9,} {true_count:>7,} {marker:>7}")

print()
# Check if the ranking matches the exact ranking
exact_top = sorted(exact.items(), key=lambda x: x[1], reverse=True)[:K]
exact_set = set(vid for vid, _ in exact_top)
heap_set = set(vid for _, vid in heap_results)
overlap = exact_set & heap_set

print(f"📊 Ranking accuracy: {len(overlap)}/{K} correct videos in top {K}")
print(f"   Processing speed: {len(view_events) / (stream_time/1000):,.0f} events/sec")

## ⚡ Performance Comparison

Let's compare the three approaches side by side:
1. **SQL all-time** — query pre-computed totals with ORDER BY + LIMIT
2. **SQL window** — GROUP BY + SUM + ORDER BY (expensive)
3. **CMS + Heap** — streaming, constant memory

In [ ]:
import matplotlib.pyplot as plt

# Benchmark each approach
results = {}

# 1. SQL all-time (pre-computed)
conn = get_db_connection()
cursor = conn.cursor()
times = []
for _ in range(50):
    start = time.time()
    cursor.execute("""
        SELECT video_id, total_views FROM video_view_totals
        ORDER BY total_views DESC LIMIT %s
    """, (K,))
    cursor.fetchall()
    times.append((time.time() - start) * 1000)
results["SQL\n(pre-computed)"] = sum(times) / len(times)

# 2. SQL window (GROUP BY + SUM)
times = []
for _ in range(50):
    start = time.time()
    cursor.execute("""
        SELECT video_id, SUM(view_count) as total FROM hourly_views
        WHERE hour_bucket >= NOW() - INTERVAL '24 hours'
        GROUP BY video_id
        ORDER BY total DESC LIMIT %s
    """, (K,))
    cursor.fetchall()
    times.append((time.time() - start) * 1000)
results["SQL\n(24h window)"] = sum(times) / len(times)
conn.close()

# 3. CMS + Heap (already measured above)
results["CMS + Heap\n(streaming)"] = stream_time

# Plot comparison
fig, ax = plt.subplots(figsize=(8, 5))
labels = list(results.keys())
values = list(results.values())
colors = ['#2ecc71', '#e74c3c', '#3498db']

bars = ax.bar(labels, values, color=colors)
ax.set_ylabel('Time (ms)')
ax.set_title(f'Top-{K} Query Performance Comparison')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1f} ms', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("💡 SQL with pre-computed totals is fastest for reads.")
print("   SQL window queries get slower with more data.")
print("   CMS + Heap processes the entire stream in one pass — no DB needed!")

## 🧪 Experiment: How Does K Affect Performance?

Let's see how changing K (the number of top items) affects the heap approach.  
Since the heap stays at size K, larger K means slightly more work per event.

In [ ]:
k_values = [5, 10, 50, 100, 500, 1000]
k_times = []

for k in k_values:
    cms = CountMinSketch(width=500, depth=5)
    topk = TopKHeap(k=k)
    
    start = time.time()
    for vid in view_events:
        cms.add(vid)
        est = cms.estimate(vid)
        topk.push(vid, est)
    elapsed = (time.time() - start) * 1000
    k_times.append(elapsed)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_values, k_times, 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('K (number of top items)')
ax.set_ylabel('Total processing time (ms)')
ax.set_title(f'CMS + Heap: Processing Time vs K ({len(view_events):,} events)')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)

for k, t in zip(k_values, k_times):
    ax.annotate(f'{t:.0f}ms', (k, t), textcoords="offset points",
                xytext=(0, 10), ha='center')

plt.tight_layout()
plt.show()

print(f"💡 Even K=1000 processes {len(view_events):,} events in {k_times[-1]:.0f} ms.")
print("   The heap adds minimal overhead because log(K) is small.")
print(f"   log₂(1000) = {10:.0f} comparisons per event — trivial!")

## 📋 Algorithm Comparison

Let's summarize when to use each approach:

In [ ]:
print("📋 Top-K Algorithm Comparison")
print("=" * 80)
print()
print(f"{'Approach':<25} {'Accuracy':<12} {'Memory':<15} {'Query Time':<15} {'Best For'}")
print("-" * 80)

approaches = [
    ("SQL (pre-computed)",   "Exact",       "O(n) on disk", "O(k)",         "All-time top-K"),
    ("SQL (window query)",   "Exact",       "O(n) on disk", "O(n log n)",   "Windowed queries (slow)"),
    ("Sort all counts",      "Exact",       "O(n) in RAM",  "O(n log n)",   "Small datasets"),
    ("CMS + Heap",           "Approximate", "O(w*d + k)",   "O(1) lookup",  "Streaming, huge scale"),
    ("Redis Sorted Set",     "Exact",       "O(n) in RAM",  "O(k + log n)", "Shared, real-time"),
]

for name, accuracy, memory, query, best_for in approaches:
    print(f"{name:<25} {accuracy:<12} {memory:<15} {query:<15} {best_for}")

print()
print("💡 In practice, production systems combine these:")
print("   • CMS + Heap for real-time streaming aggregation")
print("   • Redis Sorted Set for serving the current top-K")
print("   • SQL for historical/windowed analysis (with precomputation)")

## 🧹 Cleanup

In [ ]:
r = get_redis_client()
keys = r.keys("notebook2:*")
if keys:
    r.delete(*keys)
    print(f"🧹 Cleaned up {len(keys)} Redis keys")
else:
    print("🧹 Nothing to clean up")

## 📚 Summary

### Key Takeaways

1. **Sorting all items is O(n log n)** — too slow for billions of items
2. **A min-heap of size K** tracks the top K in O(n log K) time, O(K) memory
3. **CMS + Heap = streaming top-K** — process events one at a time, never store them all
4. **SQL with indexed ORDER BY + LIMIT** is fast for pre-computed totals (O(K) reads)
5. **Window queries in SQL** require GROUP BY + SUM and get expensive with more data

### Complexity Comparison

| Operation | Sort All | Min-Heap | CMS + Heap |
|-----------|----------|----------|------------|
| Process n events | O(n) | O(n log K) | O(n log K) |
| Memory | O(n) | O(n) counts + O(K) heap | O(w×d) sketch + O(K) heap |
| Get top K | O(n log n) | O(K log K) | O(K log K) |

### Next Up

One machine can't handle 700K events/second. In **Notebook 3**, we'll explore how to **distribute** the top-K computation across multiple machines using sharding, tumbling windows, and Redis sorted sets.